# Real-Time FinTech Fraud Detection using Spark + Quantum-Inspired Feature Selection

**Authors:** Aditya Ravi, Aayush Tiwari, Atharva Indulkar  
**Affiliation:** Department of AI & Data Science, K J Somaiya College of Engineering, Mumbai

This notebook implements a complete fraud detection pipeline with:
- Quantum-Inspired Evolutionary Algorithm (QIEA) for feature selection
- Multiple baseline feature selection methods (PCA, MI, RFE)
- Spark MLlib / sklearn classifiers (LR, RF, GBT)
- Kafka streaming simulation for real-time inference
- All figures and tables for the LaTeX report

## Cell 1: Environment Setup

Install dependencies, configure PySpark (with sklearn fallback), and import all libraries.

**Expected output:** Confirmation of PySpark initialization or sklearn fallback message.

In [ ]:
import subprocess, sys, os, time, warnings
warnings.filterwarnings('ignore')

# Install dependencies (Colab-specific)
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'pyspark==3.5.0', 'imbalanced-learn>=0.11.0'])
    os.environ.setdefault('JAVA_HOME', '/usr/lib/jvm/java-11-openjdk-amd64')

# Add project root to path (for src/ imports)
sys.path.insert(0, os.path.abspath('.'))

# Try Spark setup
SPARK_AVAILABLE = False
spark = None
try:
    from pyspark.sql import SparkSession
    from pyspark.ml.classification import (
        LogisticRegression as SparkLR,
        RandomForestClassifier as SparkRF,
        GBTClassifier as SparkGBT,
    )
    from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkScaler
    from pyspark.ml import Pipeline as SparkPipeline
    from pyspark.ml.evaluation import BinaryClassificationEvaluator
    spark = SparkSession.builder \
        .master('local[*]') \
        .appName('FraudDetection') \
        .config('spark.driver.memory', '8g') \
        .config('spark.sql.shuffle.partitions', '8') \
        .getOrCreate()
    spark.sparkContext.setLogLevel('ERROR')
    SPARK_AVAILABLE = True
    print('PySpark initialized successfully')
except Exception as e:
    print(f'PySpark unavailable ({e}), using sklearn classifiers')

# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, confusion_matrix,
)

from src.qiea import QIEAFeatureSelector
from src.preprocessing import load_data, preprocess, split_data, apply_smote
from src.evaluation import (
    compute_metrics, plot_qiea_convergence, plot_roc_curves,
    plot_confusion_matrices, plot_feature_importance,
    plot_latency_comparison, plot_streaming_latency,
    plot_system_architecture,
)

# Inline display helper
try:
    from IPython.display import Image, display as ipy_display
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False

np.random.seed(42)
RANDOM_STATE = 42


RESULTS_DIR = 'results/figures'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs('report/images', exist_ok=True)

print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'Spark available: {SPARK_AVAILABLE}')
print('Setup complete.')

## Cell 2: Data Loading & EDA

Load the Kaggle Credit Card Fraud Detection dataset. Print shape, class distribution,
class imbalance ratio, and descriptive statistics.

**Expected output:** Dataset shape (284807, 31), class counts, imbalance ratio (~577:1), summary statistics table.

In [ ]:
# Load dataset
df = load_data('data/creditcard.csv')

print(f'\nShape: {df.shape}')
print(f'\nClass Distribution:')
class_counts = df['Class'].value_counts()
print(class_counts)
print(f'\nImbalance Ratio (legitimate:fraud): {class_counts[0] / class_counts[1]:.1f}:1')

# Descriptive statistics
print('\nDescriptive Statistics:')
desc_stats = df.describe().T
try:
    display(desc_stats)
except NameError:
    print(desc_stats)

# Also load into Spark if available
if SPARK_AVAILABLE:
    spark_df = spark.read.csv('data/creditcard.csv', header=True, inferSchema=True)
    print(f'\nSpark DataFrame: {spark_df.count()} rows, {len(spark_df.columns)} columns')

## Cell 3: Data Preprocessing

- Scale `Amount` using RobustScaler (robust to outliers)
- Transform `Time` to hour-of-day: `Hour = (Time % 86400) / 3600`
- Drop original `Time` column
- Stratified train/validation/test split: 70-15-15
- Apply SMOTE to training set ONLY (target ratio 1:5 minority:majority)

**Expected output:** Feature list (30 features), split sizes, SMOTE before/after counts.

In [ ]:
# Preprocess
df_processed, feature_names = preprocess(df)
print(f'\nNumber of features: {len(feature_names)}')

# Split
print('\n--- Stratified Split ---')
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df_processed)

# SMOTE on training set only
print('\n--- SMOTE Oversampling (train only) ---')
X_train_sm, y_train_sm = apply_smote(X_train, y_train, sampling_strategy=0.2)

print(f'\nFinal shapes:')
print(f'  Train (SMOTE): {X_train_sm.shape}')
print(f'  Validation:    {X_val.shape}')
print(f'  Test:          {X_test.shape}')
print(f'\nTest set class distribution (unchanged):')
print(f'  Legitimate: {(y_test == 0).sum()}, Fraud: {(y_test == 1).sum()}')

## Cell 4: QIEA Feature Selection

Run the Quantum-Inspired Evolutionary Algorithm to select the most discriminative features.
The QIEA uses a population of qubit individuals (m=20), evolves for T=50 generations,
and selects features that maximize AUC-ROC minus a parsimony penalty.

**Expected output:** Selected features (names and count), best fitness value, runtime.

In [ ]:
print('Running QIEA Feature Selection...')
print('=' * 60)

qiea_start = time.time()

qiea = QIEAFeatureSelector(
    n_features=len(feature_names),
    pop_size=20,
    n_generations=50,
    delta_theta=0.02 * np.pi,
    lam=0.1,
    fitness_clf=RandomForestClassifier(
        n_estimators=50, max_depth=10, random_state=42, n_jobs=-1
    ),
    cv_folds=5,
    max_fitness_samples=20000,
    random_state=42,
    verbose=True,
)

qiea.fit(X_train_sm, y_train_sm)

qiea_time = time.time() - qiea_start

# Results
qiea_indices = qiea.get_selected_indices()
qiea_feature_names = [feature_names[i] for i in qiea_indices]
n_qiea = len(qiea_indices)

print(f'\n{"=" * 60}')
print(f'QIEA Runtime: {qiea_time:.1f} seconds')
print(f'Features selected: {n_qiea}/{len(feature_names)}')
print(f'Selected features: {qiea_feature_names}')
print(f'Best fitness: {qiea.best_fitness_:.4f}')

## Cell 5: Baseline Feature Selection Methods

Implement 4 baseline feature selection methods for comparison:
1. **ALL**: Use all 30 features (no selection)
2. **PCA**: Keep components explaining 95% variance
3. **Mutual Information (MI)**: Select top-k features (k = number selected by QIEA)
4. **RFE**: Recursive Feature Elimination with RF, select top-k features

**Expected output:** Features selected by each method and feature counts.

In [ ]:
k = n_qiea  # Match QIEA's feature count for fair comparison
print(f'Selecting top-{k} features for MI and RFE (matching QIEA count)\n')

# --- ALL ---
all_indices = np.arange(len(feature_names))
print(f'ALL: {len(all_indices)} features (all)')

# --- PCA ---
# Scale before PCA so the transform is consistent with Cell 6
pca_start = time.time()
pca_scaler = StandardScaler()
X_train_for_pca = pca_scaler.fit_transform(X_train_sm)
pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_for_pca)
pca_time = time.time() - pca_start
n_pca = pca.n_components_
print(f'PCA: {n_pca} components (95% variance explained, {pca_time:.1f}s)')
print(f'  Explained variance ratio (cumulative): {pca.explained_variance_ratio_.cumsum()[-1]:.4f}')

# --- Mutual Information ---
mi_start = time.time()
mi_scores = mutual_info_classif(X_train_sm, y_train_sm, random_state=RANDOM_STATE)
mi_indices = np.argsort(mi_scores)[::-1][:k]
mi_time = time.time() - mi_start
mi_feature_names = [feature_names[i] for i in mi_indices]
print(f'\nMI: {len(mi_indices)} features ({mi_time:.1f}s)')
print(f'  Selected: {mi_feature_names}')

# --- RFE ---
rfe_start = time.time()
rfe_estimator = RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1)
rfe = RFE(rfe_estimator, n_features_to_select=k, step=1)
rfe.fit(X_train_sm, y_train_sm)
rfe_indices = np.where(rfe.support_)[0]
rfe_time = time.time() - rfe_start
rfe_feature_names = [feature_names[i] for i in rfe_indices]
print(f'\nRFE: {len(rfe_indices)} features ({rfe_time:.1f}s)')
print(f'  Selected: {rfe_feature_names}')

# Summary
print(f'\n{"=" * 60}')
print('Feature Selection Summary:')
print(f'  ALL:  {len(all_indices)} features')
print(f'  PCA:  {n_pca} components')
print(f'  MI:   {len(mi_indices)} features')
print(f'  RFE:  {len(rfe_indices)} features')
print(f'  QIEA: {n_qiea} features')

## Cell 6: Model Training

Train 3 classifiers (Logistic Regression, Random Forest, GBT) with each of the 5
feature selection methods (ALL, PCA, MI, RFE, QIEA) = 15 combinations total.

For each combination:
- Standardize features
- Train on training set
- Predict on test set
- Compute Precision, Recall, F1-Score, AUC-ROC
- Measure per-sample inference latency

Uses Spark MLlib if available, otherwise sklearn with equivalent hyperparameters.

**Expected output:** Progress messages for each of the 15 training runs.

In [ ]:
# Prepare feature sets
# For index-based methods, apply StandardScaler
scaler_full = StandardScaler()
X_train_scaled = scaler_full.fit_transform(X_train_sm)
X_test_scaled = scaler_full.transform(X_test)
X_val_scaled = scaler_full.transform(X_val)

# PCA: use the same scaler fitted in Cell 5, then apply PCA
X_train_pca_scaled = pca.transform(pca_scaler.transform(X_train_sm))
X_test_pca_scaled = pca.transform(pca_scaler.transform(X_test))

feature_sets = {
    'ALL': (X_train_scaled, X_test_scaled),
    'PCA': (X_train_pca_scaled, X_test_pca_scaled),
    'MI':  (X_train_scaled[:, mi_indices], X_test_scaled[:, mi_indices]),
    'RFE': (X_train_scaled[:, rfe_indices], X_test_scaled[:, rfe_indices]),
    'QIEA': (X_train_scaled[:, qiea_indices], X_test_scaled[:, qiea_indices]),
}

# Classifier definitions (sklearn -- used as fallback or primary)
classifier_defs = {
    'LR': lambda: LogisticRegression(
        max_iter=100, C=100.0,  # C=1/regParam => regParam=0.01
        penalty='l2', solver='lbfgs', random_state=RANDOM_STATE,
        class_weight='balanced',
    ),
    'RF': lambda: RandomForestClassifier(
        n_estimators=200, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1,
        class_weight='balanced',
    ),
    'GBT': lambda: GradientBoostingClassifier(
        n_estimators=100, max_depth=8, learning_rate=0.1,
        subsample=0.8, random_state=RANDOM_STATE,
    ),
}

# Training loop
results = []
trained_models = {}  # Store for later use
predictions = {}     # Store predictions for plots

total = len(feature_sets) * len(classifier_defs)
count = 0

for fs_name, (X_tr, X_te) in feature_sets.items():
    for clf_name, clf_factory in classifier_defs.items():
        count += 1
        print(f'[{count}/{total}] Training {clf_name} with {fs_name} features '
              f'({X_tr.shape[1]} dims)...', end=' ')

        clf = clf_factory()

        # Train
        train_start = time.time()
        clf.fit(X_tr, y_train_sm)
        train_time = time.time() - train_start

        # Predict
        infer_start = time.time()
        y_pred = clf.predict(X_te)
        y_prob = clf.predict_proba(X_te)[:, 1]
        infer_time = time.time() - infer_start
        latency_ms = (infer_time / len(X_te)) * 1000  # ms per transaction

        # Metrics
        metrics = compute_metrics(y_test, y_pred, y_prob)
        metrics['FS_Method'] = fs_name
        metrics['Model'] = clf_name
        metrics['Latency_ms'] = latency_ms
        metrics['Train_Time_s'] = train_time
        metrics['N_Features'] = X_tr.shape[1]
        results.append(metrics)

        # Store model and predictions
        trained_models[(fs_name, clf_name)] = clf
        predictions[(fs_name, clf_name)] = (y_test, y_pred, y_prob)

        print(f'AUC={metrics["AUC_ROC"]:.4f}, F1={metrics["F1"]:.4f}, '
              f'Train={train_time:.1f}s, Latency={latency_ms:.4f}ms')

print(f'\nAll {total} models trained successfully.')

## Cell 7: Results Table

Display the complete results DataFrame showing all 15 model-feature combinations.
Highlights the best result per metric.

**Expected output:** Formatted table with columns: FS_Method, Model, Precision, Recall, F1, AUC_ROC, Latency_ms.

In [ ]:
results_df = pd.DataFrame(results)

# Reorder columns
col_order = ['FS_Method', 'Model', 'N_Features', 'Precision', 'Recall', 'F1',
             'AUC_ROC', 'Latency_ms', 'Train_Time_s']
results_df = results_df[col_order]

# Display
print('=' * 100)
print('COMPLETE RESULTS TABLE')
print('=' * 100)

# Format for display
display_df = results_df.copy()
for col in ['Precision', 'Recall', 'F1', 'AUC_ROC']:
    display_df[col] = display_df[col].map('{:.4f}'.format)
display_df['Latency_ms'] = display_df['Latency_ms'].map('{:.4f}'.format)
display_df['Train_Time_s'] = display_df['Train_Time_s'].map('{:.1f}'.format)

print(display_df.to_string(index=False))

# Best per metric
print(f'\n{"=" * 100}')
print('BEST RESULTS PER METRIC:')
for metric in ['Precision', 'Recall', 'F1', 'AUC_ROC']:
    best_idx = results_df[metric].astype(float).idxmax()
    best_row = results_df.loc[best_idx]
    print(f'  Best {metric}: {float(best_row[metric]):.4f} '
          f'({best_row["FS_Method"]} + {best_row["Model"]})')

## Cell 8: QIEA Convergence Plot

Plot the QIEA fitness convergence over 50 generations. Shows the best fitness line
(solid blue) and mean population fitness (shaded band).

**Expected output:** `qiea_convergence.png` saved to results/figures/.

In [ ]:
plot_qiea_convergence(
    qiea.convergence_history_,
    qiea.mean_fitness_history_,
    os.path.join(RESULTS_DIR, 'qiea_convergence.png'),
)

# Display inline
if HAS_IPYTHON:
    ipy_display(Image(filename=os.path.join(RESULTS_DIR, 'qiea_convergence.png')))

## Cell 9: ROC Curves

Overlay ROC curves for LR, RF, GBT under QIEA feature selection, plus RF with ALL
features as a dashed baseline. AUC values shown in the legend.

**Expected output:** `roc_curves.png` saved to results/figures/.

In [ ]:
roc_data = {}

# QIEA models
for clf_name in ['LR', 'RF', 'GBT']:
    key = ('QIEA', clf_name)
    if key in predictions:
        y_true, _, y_prob = predictions[key]
        roc_data[f'QIEA + {clf_name}'] = (y_true, y_prob)

# ALL + RF baseline
key_baseline = ('ALL', 'RF')
if key_baseline in predictions:
    y_true, _, y_prob = predictions[key_baseline]
    roc_data['ALL + RF (baseline)'] = (y_true, y_prob)

plot_roc_curves(
    roc_data,
    os.path.join(RESULTS_DIR, 'roc_curves.png'),
)

if HAS_IPYTHON:
    ipy_display(Image(filename=os.path.join(RESULTS_DIR, 'roc_curves.png')))


## Cell 10: Confusion Matrices

Side-by-side confusion matrices comparing the best model with ALL features vs.
the best model with QIEA features.

**Expected output:** `confusion_matrices.png` saved to results/figures/.

In [ ]:
# Find best model for ALL and QIEA by AUC-ROC
all_results = results_df[results_df['FS_Method'] == 'ALL']
best_all_idx = all_results['AUC_ROC'].idxmax()
best_all_model = results_df.loc[best_all_idx, 'Model']

qiea_results = results_df[results_df['FS_Method'] == 'QIEA']
best_qiea_idx = qiea_results['AUC_ROC'].idxmax()
best_qiea_model = results_df.loc[best_qiea_idx, 'Model']

# Get predictions
_, y_pred_all, _ = predictions[('ALL', best_all_model)]
_, y_pred_qiea, _ = predictions[('QIEA', best_qiea_model)]

cm_all = confusion_matrix(y_test, y_pred_all)
cm_qiea = confusion_matrix(y_test, y_pred_qiea)

plot_confusion_matrices(
    cm_all, cm_qiea,
    f'ALL + {best_all_model}', f'QIEA + {best_qiea_model}',
    os.path.join(RESULTS_DIR, 'confusion_matrices.png'),
)

print(f'ALL best model: {best_all_model}')
print(f'QIEA best model: {best_qiea_model}')

if HAS_IPYTHON:
    ipy_display(Image(filename=os.path.join(RESULTS_DIR, 'confusion_matrices.png')))


## Cell 11: Feature Importance Bar Chart

Horizontal bar chart showing the top 15 features by Random Forest importance.
QIEA-selected features are highlighted in green, non-selected in grey.

**Expected output:** `feature_importance.png` saved to results/figures/.

In [ ]:
# Reuse the RF model already trained on ALL features in Cell 6
rf_all_model = trained_models[('ALL', 'RF')]
importances = rf_all_model.feature_importances_

plot_feature_importance(
    feature_names,
    importances,
    qiea.best_mask_,
    os.path.join(RESULTS_DIR, 'feature_importance.png'),
    top_n=15,
)

if HAS_IPYTHON:
    ipy_display(Image(filename=os.path.join(RESULTS_DIR, 'feature_importance.png')))


## Cell 12: Latency Comparison

Grouped bar chart comparing per-transaction inference latency across all
feature selection methods and classifiers.

**Expected output:** `latency_comparison.png` saved to results/figures/.

In [ ]:
plot_latency_comparison(
    results_df,
    os.path.join(RESULTS_DIR, 'latency_comparison.png'),
)

if HAS_IPYTHON:
    ipy_display(Image(filename=os.path.join(RESULTS_DIR, 'latency_comparison.png')))


## Cell 13: Kafka Streaming Simulation

Simulate a Kafka-Spark streaming pipeline by processing test transactions in
micro-batches of 100. Measures end-to-end latency per batch and reports
average, P50, P95, P99 latencies.

**Expected output:** Latency statistics and `streaming_latency.png` saved to results/figures/.

In [ ]:
# Use the best QIEA model for streaming simulation
best_qiea_clf_name = best_qiea_model
streaming_model = trained_models[('QIEA', best_qiea_clf_name)]
X_test_qiea = X_test_scaled[:, qiea_indices]

# Micro-batch streaming simulation
BATCH_SIZE = 100
n_batches = len(X_test_qiea) // BATCH_SIZE

batch_latencies = []  # ms per batch
batch_sizes = []

print(f'Streaming simulation: {len(X_test_qiea)} transactions in '
      f'{n_batches} micro-batches of {BATCH_SIZE}')
print(f'Model: QIEA + {best_qiea_clf_name}')
print('-' * 50)

for i in range(n_batches):
    batch_start = i * BATCH_SIZE
    batch_end = batch_start + BATCH_SIZE
    batch = X_test_qiea[batch_start:batch_end]

    start = time.time()
    preds = streaming_model.predict(batch)
    probs = streaming_model.predict_proba(batch)[:, 1]
    latency_ms = (time.time() - start) * 1000

    batch_latencies.append(latency_ms)
    batch_sizes.append(len(batch))

    # Flag fraud detections
    n_fraud = preds.sum()
    if n_fraud > 0 and i % 10 == 0:
        print(f'  Batch {i+1}/{n_batches}: {n_fraud} fraud alerts, '
              f'latency={latency_ms:.2f}ms')

# Per-transaction latencies
per_txn_latencies = [lat / bs for lat, bs in zip(batch_latencies, batch_sizes)]

print(f'\n{"=" * 50}')
print('Streaming Latency Statistics (per batch):')
print(f'  Average: {np.mean(batch_latencies):.2f} ms')
print(f'  P50:     {np.percentile(batch_latencies, 50):.2f} ms')
print(f'  P95:     {np.percentile(batch_latencies, 95):.2f} ms')
print(f'  P99:     {np.percentile(batch_latencies, 99):.2f} ms')
print(f'\nPer-transaction latency:')
print(f'  Average: {np.mean(per_txn_latencies):.4f} ms')
print(f'  P95:     {np.percentile(per_txn_latencies, 95):.4f} ms')

# Plot
plot_streaming_latency(
    batch_latencies,
    batch_sizes,
    os.path.join(RESULTS_DIR, 'streaming_latency.png'),
)

if HAS_IPYTHON:
    ipy_display(Image(filename=os.path.join(RESULTS_DIR, 'streaming_latency.png')))


## Cell 14: System Architecture Diagram

Generate a system architecture diagram showing the real-time fraud detection
pipeline: Data Sources -> Kafka -> Spark Streaming -> QIEA Feature Selection -> MLlib Models -> Alerts.

**Expected output:** `system_architecture.png` saved to results/figures/.

In [ ]:
plot_system_architecture(
    os.path.join(RESULTS_DIR, 'system_architecture.png'),
)

if HAS_IPYTHON:
    ipy_display(Image(filename=os.path.join(RESULTS_DIR, 'system_architecture.png')))


## Cell 15: Summary and Export

Print the final summary of results, save the results table to CSV, and copy
all figures to the report/images/ directory.

**Expected output:** Summary statistics, `results_summary.csv`, all figures listed.

In [ ]:
import shutil

# Find overall best model
best_overall_idx = results_df['AUC_ROC'].idxmax()
best_overall = results_df.loc[best_overall_idx]

# Latency improvement: QIEA vs ALL (average across classifiers)
avg_latency_all = results_df[results_df['FS_Method'] == 'ALL']['Latency_ms'].mean()
avg_latency_qiea = results_df[results_df['FS_Method'] == 'QIEA']['Latency_ms'].mean()
latency_improvement = ((avg_latency_all - avg_latency_qiea) / avg_latency_all) * 100

print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'\nBest Model: {best_overall["FS_Method"]} + {best_overall["Model"]}')
print(f'  AUC-ROC:   {best_overall["AUC_ROC"]:.4f}')
print(f'  Precision: {best_overall["Precision"]:.4f}')
print(f'  Recall:    {best_overall["Recall"]:.4f}')
print(f'  F1-Score:  {best_overall["F1"]:.4f}')
print(f'\nQIEA Feature Selection:')
print(f'  Features selected: {n_qiea}/{len(feature_names)}')
print(f'  Selected features: {qiea_feature_names}')
print(f'  Best fitness: {qiea.best_fitness_:.4f}')
print(f'\nLatency Improvement (QIEA vs ALL):')
print(f'  ALL avg:  {avg_latency_all:.4f} ms/transaction')
print(f'  QIEA avg: {avg_latency_qiea:.4f} ms/transaction')
print(f'  Improvement: {latency_improvement:.1f}%')

# Save results to CSV
csv_path = os.path.join(RESULTS_DIR, 'results_summary.csv')
results_df.to_csv(csv_path, index=False)
print(f'\nResults saved to: {csv_path}')

# Copy figures to report/images/
figure_files = [
    'qiea_convergence.png',
    'roc_curves.png',
    'confusion_matrices.png',
    'feature_importance.png',
    'latency_comparison.png',
    'streaming_latency.png',
    'system_architecture.png',
]

print(f'\nGenerated files:')
for f in figure_files + ['results_summary.csv']:
    src = os.path.join(RESULTS_DIR, f)
    if os.path.exists(src):
        if f.endswith('.png'):
            shutil.copy2(src, os.path.join('report', 'images', f))
        print(f'  [OK] {src}')
    else:
        print(f'  [MISSING] {src}')

print(f'\nAll figures copied to report/images/')
print('\nNotebook execution complete.')

# Stop Spark session if running
if SPARK_AVAILABLE and spark is not None:
    spark.stop()
    print('Spark session stopped.')